In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

In [2]:
def load_json(file_path):
    with open(file_path, 'r') as f:
        return json.load(f)

In [4]:
root = '../data/'

filah_data = load_json(root + 'FILAH.json')
trout_data = load_json(root + 'TROUT.json')
journalist_data = load_json(root + 'journalist.json')
road_map_data = load_json(root + 'road_map.json')

# Attempt 1

In [5]:
def preprocess_kb(kb):
    nodes = pd.DataFrame(kb['nodes'])
    links = pd.DataFrame(kb['links'])

    unique_node_types = nodes['type'].unique()
    entities_ids = {}
    for node_type in unique_node_types:  
        if str(node_type) == 'nan':
            node_type_df = nodes[nodes['type'].isna()]
        else:
            node_type_df = nodes[nodes['type'] == node_type]
        entities_ids[str(node_type)] = node_type_df['id'].unique().tolist()

    def get_entity_type_from_id(id):
        for node_type in entities_ids:
            if id in entities_ids[node_type]:
                return node_type
        return None

    source_types = []
    target_types = []
    for link in links.itertuples():
        source = link.source
        target = link.target

        source_type = get_entity_type_from_id(source)
        target_type = get_entity_type_from_id(target)

        source_types.append(source_type)
        target_types.append(target_type)

    links['source_type'] = source_types
    links['target_type'] = target_types

    return nodes, links

In [ ]:
nodes, links = preprocess_kb(journalist_data)

In [ ]:
nodes[nodes['type'] == 'entity.person']

In [104]:
person_id = 'Seal'
# person_id = 'Carol Limpet'

In [105]:
sentiments = links[(links['target'] == person_id) & (links['sentiment'].notna())].explode('industry').drop_duplicates(subset=['sentiment', 'reason'])

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(sentiments['industry'].fillna('None'), sentiments['sentiment'], alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.ylim(-1, 1)
plt.xlabel('Industry')
plt.ylabel('Sentiment')
plt.title('Sentiment by Industry')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
trips = links[(links['source_type'] == 'trip') & (links['target'] == person_id)].rename(columns={
    'source': 'trip_id',
    'target': 'person_id',
})
trips = trips.merge(
    nodes.loc[nodes['type'] == 'trip', ['id', 'date', 'start', 'end']].rename(columns={'id': 'trip_id'}),
    on='trip_id'
)
trips.head()

In [ ]:
trips_places = links[(links['source'].isin(trips['trip_id'])) & (links['target_type'] == 'place')].rename(columns={
    'source': 'trip_id',
    'target': 'place_id',
})
trips_places = trips_places.merge(
    nodes.loc[nodes['id'].isin(trips_places['place_id']), ['id', 'zone', 'zone_detail']].rename(columns={'id': 'place_id'}), 
    on='place_id'
)
trips_places['year_month'] = trips_places['time'].apply(lambda x: x.split('-')[0] + '-' + x.split('-')[1]).astype(str)
trips_places.head()

In [ ]:
trips_plot = trips_places.groupby(['year_month', 'zone']).agg({'place_id': 'count'}).reset_index()
trips_plot.head()

In [110]:
unique_dates = trips_plot['year_month'].unique()
unique_zones = trips_plot['zone'].unique()
time_series = np.zeros((len(unique_dates), len(unique_zones)))
for trip in trips_plot.itertuples():
    time_series[np.where(unique_dates == trip.year_month)[0][0], 
                np.where(unique_zones == trip.zone)[0][0]] = trip.place_id

In [ ]:
time_series = np.cumsum(time_series, axis=0)
time_series = time_series / time_series.sum(axis=1).reshape(-1, 1)
time_series

In [ ]:
# Create a line plot of the trips by year_month and zone using time_series
plt.figure(figsize=(12, 6))
for i, zone in enumerate(unique_zones):
    plt.plot(unique_dates, time_series[:, i], label=zone)
plt.xlabel('Year-Month')
plt.ylabel('Percentage of Total Trips')
plt.title('Trips by Year-Month and Zone')
plt.legend(title='Zone')

# Attempt 2

In [5]:
class KB:
    def __init__(self, kb_path):
        kb = self.load_json(kb_path)
        self.nodes, self.links = self.preprocess_kb(kb)

    def load_json(self, file_path):
        with open(file_path, 'r') as f:
            return json.load(f)

    def preprocess_kb(self, kb):
        nodes = pd.DataFrame(kb['nodes'])
        links = pd.DataFrame(kb['links'])

        nodes['type'] = nodes['type'].fillna('nan')

        unique_node_types = nodes['type'].unique()
        entities_ids = {}
        for node_type in unique_node_types: 
            node_type_df = nodes[nodes['type'] == node_type]
            entities_ids[str(node_type)] = node_type_df['id'].unique().tolist()

        def get_entity_type_from_id(id):
            for node_type in entities_ids:
                if id in entities_ids[node_type]:
                    return node_type
            return None

        source_types = []
        target_types = []
        for link in links.itertuples():
            source = link.source
            target = link.target

            source_type = get_entity_type_from_id(source)
            target_type = get_entity_type_from_id(target)

            source_types.append(source_type)
            target_types.append(target_type)

        links['source_type'] = source_types
        links['target_type'] = target_types
        return nodes, links
    
    def get_nodes_type(self, node_type):
        return self.nodes[self.nodes['type'] == node_type].copy().dropna(axis=1, how='all')
    
    def get_edges_type(self, edge_type, source_nodes = None, target_nodes = None):
        filtered_links = self.links[
            (self.links['source_type'] == edge_type[0]) &
            (self.links['target_type'] == edge_type[1])
        ]
        if source_nodes:
            filtered_links = filtered_links[filtered_links['source'].isin(source_nodes)]
        if target_nodes:
            filtered_links = filtered_links[filtered_links['target'].isin(target_nodes)]
        return filtered_links.copy().dropna(axis=1, how='all')

    def get_parents(self, edge_type, node_values = None):
        filtered_links = self.links[
            (self.links['source_type'] == edge_type[0]) &
            (self.links['target_type'] == edge_type[1])
        ]
        if node_values:
            filtered_links = filtered_links[filtered_links['target'].isin(node_values)]
        return nodes[nodes['id'].isin(filtered_links['source'])].copy().dropna(axis=1, how='all')
    
    def get_children(self, edge_type, node_values = None):
        filtered_links = self.links[
            (self.links['source_type'] == edge_type[0]) &
            (self.links['target_type'] == edge_type[1])
        ]
        if node_values:
            filtered_links = filtered_links[filtered_links['source'].isin(node_values)]
        return nodes[nodes['id'].isin(filtered_links['target'])].copy().dropna(axis=1, how='all')

In [7]:
root = '../data'

In [8]:
journalist_kb = KB(os.path.join(root, 'journalist.json'))
filah_kb = KB(os.path.join(root, 'FILAH.json'))
trout_kb = KB(os.path.join(root, 'TROUT.json'))

In [9]:
topics_map = {
    'Expanding the tourist wharf/infrastructure in Haacklee': 'tourism',
    'Statue to honor John Smoth in Port Grove': 'tourism',
    'Deep maintenance to the commercial fishing dock at Himark': 'fishing',
    'Money for a new crane at Lomark': 'fishing',
    'Fish Vacuum': 'fishing',
    'Low-volume unload crane in Haacklee': 'fishing',
    'Affordable housing for fishing workers (conflict with short-term rentals)': 'fishing',
    'Renaming a park in Himark': 'tourism',
    'Putting a name on pictureque harbor area in Port Grove': 'tourism',
    'Putting a name on the inspection office in Lomark': 'tourism',
    'Establishing a Marine Life Observation Deck in Centralia': 'tourism',
    'Hosting an Annual Seafood Festival in Paackland': 'tourism',
    'Developing a Heritage Walking Tour in Haacklee': 'tourism',
    'Establishing a Waterfront Market in Haacklee': 'fishing',
    'Plan for Concert': 'tourism',
}

In [10]:
kb = journalist_kb

n_topics = kb.get_nodes_type('topic')
n_topics['group'] = n_topics['long_topic'].map(topics_map)
n_persons = kb.get_nodes_type('entity.person')
n_discussions = kb.get_nodes_type('discussion')
n_plans = kb.get_nodes_type('plan').merge(
    kb.get_edges_type(('plan', 'topic'))[['source', 'target']],
    left_on='id',
    right_on='source'
).rename(columns={'target': 'topic'}).drop(columns=['source'])

e_discussions_plans = kb.get_edges_type(('discussion', 'plan'))
e_discussions_plans['status'] = e_discussions_plans['status'].str.lower()
e_discussions_persons = kb.get_edges_type(('discussion', 'entity.person'))
e_plans_persons = kb.get_edges_type(('plan', 'entity.person'))

In [17]:
kb.get_edges_type(('discussion', 'plan'))['status'].str.lower().unique()

array(['introduced', 'completed', 'planned', 'in_progress'], dtype=object)

In [64]:
n_discussions = n_discussions.merge(
    e_discussions_plans[['source', 'status']],
    left_on='id',
    right_on='source',
    how='left'
).drop(columns=['source'])

In [65]:
e_discussions_plans = e_discussions_plans.rename(columns={
    'target': 'plan_id',
    'source': 'discussion_id'
}).merge(
    kb.get_edges_type(('plan', 'topic'))[['source', 'target']],
    left_on='plan_id',
    right_on='source'
).rename(columns={'target': 'topic_id'}).drop(columns=['source'])

In [66]:
n_plans['plan_type'] = n_plans['plan_type'].str.lower()
e_plans_persons = e_plans_persons.merge(
    n_plans[['id', 'plan_type']],
    left_on='source',
    right_on='id',
    how='left'
).drop(columns=['id'])

In [67]:
e_discussions_plans = e_discussions_plans.merge(
    n_plans[['id', 'plan_type']],
    left_on='plan_id',
    right_on='id',
    how='left'
).drop(columns=['id'])

In [13]:
# Iterate over topic, creating id for each plan
for topic_id in e_discussions_plans['topic_id'].unique():
    topic_plans = e_discussions_plans[e_discussions_plans['topic_id'] == topic_id]
    plan_order_map = {plan_id: i for i, plan_id in enumerate(topic_plans.drop_duplicates(subset=['plan_id'])['plan_id'].values)}
    e_discussions_plans.loc[
        topic_plans.index,
        'plan_order'
    ] = e_discussions_plans.loc[
        topic_plans.index,
        'plan_id'
    ].map(plan_order_map)
e_discussions_plans['plan_order'] = e_discussions_plans['plan_order'].astype(int)

# Iterate over plan, creating id for each discussion
for plan_id in e_discussions_plans['plan_id'].unique():
    plan_discussions = e_discussions_plans[e_discussions_plans['plan_id'] == plan_id]
    discussion_order_map = {discussion_id: i for i, discussion_id in enumerate(plan_discussions.drop_duplicates(subset=['discussion_id'])['discussion_id'].values)}
    e_discussions_plans.loc[
        plan_discussions.index,
        'discussion_order'
    ] = e_discussions_plans.loc[
        plan_discussions.index,
        'discussion_id'
    ].map(discussion_order_map)
e_discussions_plans['discussion_order'] = e_discussions_plans['discussion_order'].astype(int)

In [14]:
for df in [e_discussions_persons, e_plans_persons]:
    df['sentiment'] = df['sentiment'].fillna(0)
    df['reason'] = df['reason'].fillna('')
    df['industry'] = df['industry'].apply(lambda x: [] if x is None else x)

In [15]:
data_discussions = []
for person_id in n_persons['id'].values:
    for topic_id in n_topics['id'].values:
        plans_topic = n_plans[n_plans['topic'] == topic_id]
        e_discussions_plans_topic = e_discussions_plans[e_discussions_plans['plan_id'].isin(plans_topic['id'].tolist())]
        for discussion_id, plan_id, discussion_order, plan_order, status in e_discussions_plans_topic[['discussion_id', 'plan_id', 'discussion_order', 'plan_order', 'status']].values:
            discussion_info = e_discussions_persons[(e_discussions_persons['source'] == discussion_id) & (e_discussions_persons['target'] == person_id)]

            data_square = {
                'person_id': person_id,
                'topic_id': topic_id,
                'plan_id': plan_id,
                'plan_order': plan_order,
                'discussion_id': discussion_id,
                'discussion_order': discussion_order,
                'discussion_info': discussion_info.to_dict(orient='records'),
                'discussion_status': status,
                'absent': discussion_info.empty,
            }
            data_discussions.append(data_square)

In [16]:
data_plans = []
for person_id in n_persons['id'].values:
    for topic_id in n_topics['id'].values:
        plans_topic = n_plans[n_plans['topic'] == topic_id]
        e_discussions_plans_topic = e_discussions_plans[e_discussions_plans['plan_id'].isin(plans_topic['id'].tolist())]
        e_discussions_plans_topic = e_discussions_plans_topic.drop_duplicates(subset=['plan_id'])
        for plan_id, plan_order, plan_type in e_discussions_plans_topic[['plan_id', 'plan_order', 'plan_type']].values:
            plan_info = e_plans_persons[(e_plans_persons['source'] == plan_id) & (e_plans_persons['target'] == person_id)]

            data_square = {
                'person_id': person_id,
                'topic_id': topic_id,
                'plan_id': plan_id,
                'plan_order': plan_order,
                'plan_type': plan_type,
                'plan_info': plan_info.to_dict(orient='records'),
                'absent': plan_info.empty,
            }
            data_plans.append(data_square)

In [17]:
data_persons = n_persons.to_dict(orient='records')

In [18]:
data_topics = n_topics.sort_values(by='group').to_dict(orient='records')

In [19]:
with open('data.json', 'w') as f:
    json.dump({
        'discussions': data_discussions,
        'plans': data_plans,
        'persons': data_persons,
        'topics': data_topics
    }, f, indent=4)

# Testing

In [18]:
kb = journalist_kb

n_meetings = kb.get_nodes_type('meeting')
e_meetings_plans = kb.get_edges_type(('meeting', 'plan'))
e_meetings_discussions = kb.get_edges_type(('meeting', 'discussion'))

e_discussions_places = kb.get_edges_type(('discussion', 'place'))
e_plans_places = kb.get_edges_type(('plan', 'place'))

n_places = kb.get_nodes_type('place')

In [19]:
discussions_places = e_discussions_places.merge(
    n_places[['id', 'name']], 
    left_on='target', 
    right_on='id', how='left'
).drop(columns=['id']).merge(
    e_meetings_discussions[['source', 'target']].rename(columns={'source': 'meeting_id', 'target': 'discussion_id'}),
    left_on='source',
    right_on='discussion_id',
    how='left'
)[['meeting_id', 'name']]

plans_places = e_plans_places.merge(
    n_places[['id', 'name']], 
    left_on='target', 
    right_on='id', how='left'
).drop(columns=['id']).merge(
    e_meetings_plans[['source', 'target']].rename(columns={'source': 'meeting_id', 'target': 'plan_id'}),
    left_on='source',
    right_on='plan_id',
    how='left'
)[['meeting_id', 'name']]

In [21]:
pd.concat([plans_places, discussions_places]).drop_duplicates(subset=['meeting_id', 'name']).sort_values(by='meeting_id')

,meeting_id,name
1,Meeting_10,The Bait & Stich
3,Meeting_11,The Bait & Stich
5,Meeting_11,Tropics Environmental Hub
7,Meeting_12,The Bait & Stich
11,Meeting_12,Tropics Environmental Hub
6,Meeting_2,Dock & Roll Hall of Fame
13,Meeting_3,Dock & Roll Hall of Fame
1,Meeting_4,Anchor Way Grill
2,Meeting_5,Himark City Hall
4,Meeting_8,Lomark Civic Plaza
